# Bronze para Prata: Construção de Tabela Única

Este notebook apresenta os códigos produzidos para a padronização e integração de dados de duas tabelas distintas ( `tabela_tiroteios_bronze` e `tabela_bairros_bronze` ) para a confecção do MVP em Engenharia de Dados - PUC Rio.

In [0]:
%sql
SELECT
  typeof(max(Data_Horario_Tiroteio)) AS original_type,
  'dd/MM/yyyy HH:mm:ss' AS detected_pattern,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN to_timestamp(Data_Horario_Tiroteio, 'dd/MM/yyyy HH:mm:ss') IS NULL AND Data_Horario_Tiroteio IS NOT NULL THEN 1 ELSE 0 END) AS invalid_format_rows
FROM tabela_tiroteios_bronze;

In [0]:
%sql
CREATE OR REPLACE TABLE tabela_tiroteios_prata AS
SELECT
  * EXCEPT (Data_Horario_Tiroteio),
  to_timestamp(Data_Horario_Tiroteio, 'dd/MM/yyyy HH:mm:ss') AS Data_Horario_Tiroteio
FROM tabela_tiroteios_bronze;

In [0]:
%sql
SELECT
  typeof(max(Data_Horario_Tiroteio)) AS converted_type,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN Data_Horario_Tiroteio IS NULL THEN 1 ELSE 0 END) AS null_timestamps_after_conversion,
  MIN(Data_Horario_Tiroteio) AS min_timestamp,
  MAX(Data_Horario_Tiroteio) AS max_timestamp
FROM tabela_tiroteios_prata;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.tabela_tiroteios_bairros_prata AS
WITH bairros_normalized AS (
  SELECT
    Nome_Bairro,
    AP_Bairro,
    RA_Bairro,
    RP_Bairro,
    COD_BAIRRO,
    Total_Pop_Bairro_2022,
    upper(translate(trim(Nome_Bairro), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ', 'AAAAAEEEEIIIIOOOOOUUUUC')) AS bairro_key
  FROM workspace.default.tabela_bairros_bronze
),
-- Manual mapping for entries that cannot be matched automatically:
--   FREGUESIA (JACAREPAGUA) -> FREGUESIA in AP 4 (Zona Oeste / Jacarepagua)
--   FREGUESIA (ILHA)        -> FREGUESIA in AP 3 (Ilha do Governador)
--   RICARDO DE ALBUQUERQUE  -> RICADO DE ALBUQUERQUE (typo in bairros table)
--   NI (nao identificado)   -> left unmatched (no bairro equivalent)
manual_mapping AS (
  SELECT 'FREGUESIA (JACAREPAGUA)' AS tiroteio_nome, AP_Bairro, RP_Bairro, COD_BAIRRO, Total_Pop_Bairro_2022
  FROM workspace.default.tabela_bairros_bronze WHERE Nome_Bairro = 'FREGUESIA' AND AP_Bairro = 4
  UNION ALL
  SELECT 'FREGUESIA (ILHA)', AP_Bairro, RP_Bairro, COD_BAIRRO, Total_Pop_Bairro_2022
  FROM workspace.default.tabela_bairros_bronze WHERE Nome_Bairro = 'FREGUESIA' AND AP_Bairro = 3
  UNION ALL
  SELECT 'RICARDO DE ALBUQUERQUE', AP_Bairro, RP_Bairro, COD_BAIRRO, Total_Pop_Bairro_2022
  FROM workspace.default.tabela_bairros_bronze WHERE Nome_Bairro = 'RICADO DE ALBUQUERQUE'
)
SELECT
  t.*,
  COALESCE(b.AP_Bairro,    m.AP_Bairro)    AS AP_Bairro,
  COALESCE(b.RP_Bairro,    m.RP_Bairro)    AS RP_Bairro,
  COALESCE(b.COD_BAIRRO,   m.COD_BAIRRO)   AS COD_BAIRRO,
  COALESCE(b.Total_Pop_Bairro_2022, m.Total_Pop_Bairro_2022) AS Total_Pop_Bairro_2022
FROM workspace.default.tabela_tiroteios_prata t
LEFT JOIN bairros_normalized b
  ON upper(translate(trim(t.Nome_Bairro), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ', 'AAAAAEEEEIIIIOOOOOUUUUC')) = b.bairro_key
LEFT JOIN manual_mapping m
  ON t.Nome_Bairro = m.tiroteio_nome
WHERE upper(trim(t.Nome_Bairro)) <> 'NI' OR t.Nome_Bairro IS NULL;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.default.tabela_tiroteios_prata) AS base_rows,
  COUNT(*) AS merged_rows,
  SUM(CASE WHEN AP_Bairro IS NULL THEN 1 ELSE 0 END) AS rows_without_bairro_match,
  COUNT(DISTINCT Nome_Bairro) AS distinct_bairros_in_merged_table,
  COUNT(DISTINCT CASE WHEN AP_Bairro IS NULL THEN Nome_Bairro END) AS distinct_unmatched_bairros
FROM workspace.default.tabela_tiroteios_bairros_prata;

In [0]:
%sql
SELECT
  Nome_Bairro,
  COUNT(*) AS total_tiroteios,
  MIN(Data_Horario_Tiroteio) AS primeira_ocorrencia,
  MAX(Data_Horario_Tiroteio) AS ultima_ocorrencia
FROM workspace.default.tabela_tiroteios_bairros_prata
WHERE AP_Bairro IS NULL
  AND Nome_Bairro IS NOT NULL
GROUP BY Nome_Bairro
ORDER BY total_tiroteios DESC;

In [0]:
%sql
use catalog `workspace`; select * from `default`.`tabela_tiroteios_bairros_prata` limit 100;

In [0]:
%sql
ALTER TABLE workspace.default.tabela_tiroteios_bairros_prata
ADD COLUMNS (
  Dia_Tiroteio INT,
  Mes_Tiroteio INT,
  Ano_Tiroteio INT
);

UPDATE workspace.default.tabela_tiroteios_bairros_prata
SET
  Dia_Tiroteio = day(Data_Tiroteio),
  Mes_Tiroteio = month(Data_Tiroteio),
  Ano_Tiroteio = year(Data_Tiroteio);

In [0]:
%sql
SELECT
  Data_Tiroteio,
  Dia_Tiroteio,
  Mes_Tiroteio,
  Ano_Tiroteio
FROM workspace.default.tabela_tiroteios_bairros_prata
WHERE Data_Tiroteio IS NOT NULL
ORDER BY Data_Tiroteio
LIMIT 5;

In [0]:
%sql
DELETE FROM workspace.default.tabela_tiroteios_bairros_prata
WHERE Ano_Tiroteio = 2019;

In [0]:
%sql
SELECT
  COUNT(*) AS remaining_2019_rows,
  MIN(Data_Tiroteio) AS min_remaining_date,
  MAX(Data_Tiroteio) AS max_remaining_date,
  COUNT(*) FILTER (WHERE Ano_Tiroteio IS NULL) AS null_year_rows,
  COUNT(*) AS total_rows_checked
FROM workspace.default.tabela_tiroteios_bairros_prata
WHERE Ano_Tiroteio = 2019 OR Ano_Tiroteio IS NULL;

In [0]:
%sql
ALTER TABLE workspace.default.tabela_tiroteios_bairros_prata
ADD COLUMNS (
  trimestre INT,
  semestre INT
);

UPDATE workspace.default.tabela_tiroteios_bairros_prata
SET
  trimestre = quarter(Data_Tiroteio),
  semestre = CASE
    WHEN month(Data_Tiroteio) BETWEEN 1 AND 6 THEN 1
    WHEN month(Data_Tiroteio) BETWEEN 7 AND 12 THEN 2
    ELSE NULL
  END;

In [0]:
%sql
SELECT
  Data_Tiroteio,
  Mes_Tiroteio,
  trimestre,
  semestre
FROM workspace.default.tabela_tiroteios_bairros_prata
WHERE Data_Tiroteio IS NOT NULL
ORDER BY Data_Tiroteio
LIMIT 500;